In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
fake = pd.read_csv('Fake.csv')
true = pd.read_csv('True.csv')
fake['label'] = 0  # 0 = fake
true['label'] = 1  # 1 = real

In [3]:
df = pd.concat([fake, true], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

In [4]:
df['text_combined'] = df['title'] + ' ' + df['text']
df = df.dropna(subset=['text_combined'])
 
print(f"Total articles: {len(df)} | Fake: {(df['label']==0).sum()} | Real: {(df['label']==1).sum()}")

Total articles: 44898 | Fake: 23481 | Real: 21417


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text_combined'], df['label'], test_size=0.2, random_state=42, stratify=df['label'])
 
# --- TF-IDF vectorization ---
tfidf = TfidfVectorizer(max_features=10000, stop_words='english', ngram_range=(1, 2))
X_train_tf = tfidf.fit_transform(X_train)
X_test_tf  = tfidf.transform(X_test)
 
# --- train logistic regression ---
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_tf, y_train)
 
y_pred = model.predict(X_test_tf)
 
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=['Fake', 'Real']))



Accuracy: 0.9884
              precision    recall  f1-score   support

        Fake       0.99      0.99      0.99      4696
        Real       0.98      0.99      0.99      4284

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980



In [6]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Predicted Fake', 'Predicted Real'], fontsize=11)
ax.set_yticklabels(['Actually Fake', 'Actually Real'], fontsize=11)
ax.set_title('Confusion Matrix', fontsize=13, fontweight='bold')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i,j]:,}', ha='center', va='center', fontsize=16,
                color='white' if cm[i,j] > cm.max()/2 else 'black', fontweight='bold')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('fig1_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.close()

In [7]:
coefs = model.coef_[0]
feat_names = tfidf.get_feature_names_out()
top_n = 15
top_fake_idx = np.argsort(coefs)[:top_n]
top_real_idx = np.argsort(coefs)[-top_n:][::-1]
 
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
fig.suptitle('Top 15 TF-IDF Features by Class', fontsize=13, fontweight='bold')
axes[0].barh(range(top_n), coefs[top_fake_idx], color='#e74c3c', alpha=0.8)
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels([feat_names[i] for i in top_fake_idx], fontsize=9)
axes[0].set_xlabel('Coefficient'); axes[0].set_title('Fake News Predictors', fontweight='bold')
axes[0].invert_yaxis()
axes[1].barh(range(top_n), coefs[top_real_idx], color='#2ecc71', alpha=0.8)
axes[1].set_yticks(range(top_n))
axes[1].set_yticklabels([feat_names[i] for i in top_real_idx], fontsize=9)
axes[1].set_xlabel('Coefficient'); axes[1].set_title('Real News Predictors', fontweight='bold')
axes[1].invert_yaxis()
plt.tight_layout()
plt.savefig('fig2_top_features.png', dpi=150, bbox_inches='tight')
plt.close()

In [8]:
from sklearn.metrics import precision_recall_fscore_support
p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, labels=[0, 1])
x = np.arange(3); width = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - width/2, [p[0], r[0], f1[0]], width, label='Fake', color='#e74c3c', alpha=0.85)
b2 = ax.bar(x + width/2, [p[1], r[1], f1[1]], width, label='Real', color='#2ecc71', alpha=0.85)
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)
ax.set_ylim(0.95, 1.005); ax.set_xticks(x)
ax.set_xticklabels(['Precision', 'Recall', 'F1-Score'], fontsize=12)
ax.set_ylabel('Score'); ax.set_title('Model Performance by Class', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig3_metrics.png', dpi=150, bbox_inches='tight')
plt.close()
 

In [9]:
test_df = X_test.reset_index(drop=True).to_frame()
test_df['true_label'] = y_test.reset_index(drop=True)
test_df['pred_label'] = y_pred
test_df['title'] = df.loc[X_test.index, 'title'].reset_index(drop=True)
test_df['text']  = df.loc[X_test.index, 'text'].reset_index(drop=True)
 
wrong = test_df[test_df['true_label'] != test_df['pred_label']]
print(f"\nTotal misclassified: {len(wrong)} out of {len(test_df)}")
lmap = {0: 'Fake', 1: 'Real'}
for i, (_, row) in enumerate(wrong.head(5).iterrows()):
    print(f"\n--- Wrong #{i+1} ---")
    print(f"True: {lmap[row['true_label']]}  |  Predicted: {lmap[row['pred_label']]}")
    print(f"Title: {row['title'][:150]}")
    print(f"Snippet: {row['text'][:250]}")
 



Total misclassified: 104 out of 8980

--- Wrong #1 ---
True: Fake  |  Predicted: Real
Title: WHY REUTERS IS SAYING With “reasonable confidence” A Republican Will Win The White House In 2016
Snippet: After examining the data Reuters used to make their bold prediction, we noticed they hadn t given any consideration to the high probability that Democrats will likely find a way to give illegal aliens the opportunity to cast their ballots in 2016. Il

--- Wrong #2 ---
True: Fake  |  Predicted: Real
Title: How Is Panama’s “Migrant Crisis” Giving A FREE PASS TO THE U.S. To Anyone From Asia, Cuba, Africa, Haiti…When Did We Become The World’s Dumping Ground
Snippet: Let this sink in The word has been out for some time now that the US borders are open. Now we have the President of Panama giving an even bigger free pass to ANYONE  to come to America. We are going to be Europe soon Obama is making sure of it Panama

--- Wrong #3 ---
True: Fake  |  Predicted: Real
Title: How Is Panama’s “Migrant C